# Q-MicroSE-SVQC: Parameter-Matched Quantum Channel Gating for Lightweight Speech Enhancement

This notebook is a research-oriented rebuild of the earlier Q-MicroSE notebook. It replaces the trigonometric "quantum-style" proxy with an **exact differentiable state-vector simulator** that applies actual unitary quantum gates and measures Pauli expectation values.

The controlled comparison contains three arms:

1. **Baseline:** shared lightweight speech-enhancement backbone without a conditioner.
2. **MLP:** the same backbone with a parameter-matched classical latent mixer.
3. **QNN:** the same backbone with a genuine 6-qubit variational quantum circuit (VQC).

The MLP and QNN conditioners have **exactly the same trainable parameter count**. Both receive the same six input features, generate the same twelve output statistics, use the same projections, and are trained with the same data, losses, checkpoints, and evaluation manifest.

> Important scientific rule: the notebook is designed to give the quantum model a fair opportunity, not to force it to win. A paper should claim a quantum benefit only when the QNN exceeds the parameter-matched MLP across multiple seeds and paired statistical tests.

## 0 — Optional package installation

The main quantum model does **not** require PennyLane. Its exact state-vector simulator is implemented directly in PyTorch, making it differentiable and GPU-compatible. PennyLane is used only in an optional verification cell.

In [ ]:
# Run in a fresh Colab environment if needed.
# %pip install -q librosa==0.11.0 soundfile pandas tqdm scipy pesq pystoi matplotlib
# Optional independent circuit verification:
# %pip install -q pennylane==0.45.0 pennylane-lightning==0.45.0

In [ ]:
print("Hello")

## 1 — Imports, configuration, and reproducibility

In [ ]:
import os
import gc
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from contextlib import nullcontext
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore", category=UserWarning)

@dataclass
class Config:
    # Audio
    sample_rate: int = 16_000
    win_length: int = 320
    hop_length: int = 160
    n_fft: int = 320
    crop_seconds: int = 8

    # Data
    dataset_root: str = "my_speech_enhancement_dataset_v3"
    batch_size: int = 8
    num_workers: int = 2
    balance_snr: bool = True

    # Shared backbone
    channels: int = 72
    num_blocks: int = 7
    conditioner_after_block: int = 3
    dropout: float = 0.10
    phase_residual_scale: float = 0.75
    complex_residual_scale: float = 0.10

    # Genuine quantum circuit
    n_qubits: int = 6
    q_layers: int = 2
    entanglement: str = "ring"       # ring, linear, none
    data_reuploading: bool = True

    # Optimisation
    shared_pretrain_epochs: int = 20
    finetune_epochs: int = 35
    early_stopping_patience: int = 6
    min_delta: float = 1e-4
    backbone_lr: float = 1e-4
    conditioner_lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0
    use_amp: bool = True

    # Experiment
    experiment_name: str = "qmicrose_svqc_026m"
    seeds: tuple = (17, 42, 123)
    variants: tuple = ("baseline", "mlp", "qnn")
    strict_determinism: bool = True
    resume: bool = True

    # Evaluation
    snr_values: tuple = (-5, 0, 5, 10, 15, 20)
    max_test_files_per_snr: int | None = None  # None = evaluate every file
    metric_workers: int = 4
    bootstrap_repeats: int = 2000

CFG = Config()
CFG.crop_length = CFG.sample_rate * CFG.crop_seconds
CFG.freq_bins = CFG.n_fft // 2 + 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(CFG.use_amp and DEVICE.type == "cuda")


def seed_everything(seed: int, deterministic: bool = CFG.strict_determinism):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic
    try:
        torch.use_deterministic_algorithms(deterministic, warn_only=True)
    except Exception:
        pass


def seed_worker(worker_id: int):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def sha256_file(path: str | Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

seed_everything(CFG.seeds[0])
torch.set_float32_matmul_precision("high")

print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(json.dumps(asdict(CFG), indent=2, default=str))

## 2 — Dataset validation, locked manifests, and leakage audit

In [ ]:
REQUIRED_COLUMNS = {"noisy_path", "clean_path"}


def resolve_audio_path(value: str, csv_path: str | Path, dataset_root: str | Path) -> str:
    """Resolve absolute paths and common relative-path conventions robustly."""
    p = Path(str(value)).expanduser()
    if p.is_absolute():
        return str(p)

    csv_path = Path(csv_path).resolve()
    candidates = [
        csv_path.parent / p,
        Path(dataset_root).resolve() / p,
        Path.cwd() / p,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    # Return the most natural candidate so the error message is informative.
    return str(candidates[0].resolve())


def load_metadata(csv_path: str | Path, dataset_root: str | Path = CFG.dataset_root) -> pd.DataFrame:
    csv_path = Path(csv_path)
    frame = pd.read_csv(csv_path)
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"{csv_path} is missing required columns: {sorted(missing)}")

    frame = frame.copy()
    frame["noisy_path"] = frame["noisy_path"].map(lambda p: resolve_audio_path(p, csv_path, dataset_root))
    frame["clean_path"] = frame["clean_path"].map(lambda p: resolve_audio_path(p, csv_path, dataset_root))
    if "snr_db" not in frame.columns:
        frame["snr_db"] = np.nan

    frame["sample_id"] = [
        hashlib.sha1(f"{n}|{c}|{s}".encode()).hexdigest()[:16]
        for n, c, s in zip(frame["noisy_path"], frame["clean_path"], frame["snr_db"])
    ]
    return frame


def validate_file_existence(frame: pd.DataFrame, name: str, check_all: bool = True):
    inspect = frame if check_all else frame.head(100)
    missing_noisy = [p for p in inspect["noisy_path"] if not Path(p).exists()]
    missing_clean = [p for p in inspect["clean_path"] if not Path(p).exists()]
    if missing_noisy or missing_clean:
        raise FileNotFoundError(
            f"{name}: missing {len(missing_noisy)} noisy and {len(missing_clean)} clean files. "
            f"Examples: {(missing_noisy + missing_clean)[:5]}"
        )
    print(f"{name}: {len(frame):,} rows; all inspected paths exist.")


def audit_split_leakage(split_frames: dict[str, pd.DataFrame]):
    """Detect exact clean-utterance and sample-ID overlap across splits."""
    names = list(split_frames)
    findings = []
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            left_clean = set(split_frames[left]["clean_path"].map(lambda p: str(Path(p).resolve())))
            right_clean = set(split_frames[right]["clean_path"].map(lambda p: str(Path(p).resolve())))
            clean_overlap = left_clean & right_clean
            id_overlap = set(split_frames[left]["sample_id"]) & set(split_frames[right]["sample_id"])
            findings.append({
                "split_a": left,
                "split_b": right,
                "clean_path_overlap": len(clean_overlap),
                "sample_id_overlap": len(id_overlap),
            })
    report = pd.DataFrame(findings)
    display(report)
    if (report[["clean_path_overlap", "sample_id_overlap"]].to_numpy() > 0).any():
        raise RuntimeError(
            "Data leakage detected. Do not train or publish until exact clean utterances are split-disjoint."
        )
    print("No exact clean-path or sample-ID overlap was detected.")


def prepare_manifests(dataset_root: str | Path = CFG.dataset_root):
    root = Path(dataset_root)
    frames = {}
    for split in ("train", "val", "test"):
        csv_path = root / split / "metadata.csv"
        if not csv_path.exists():
            raise FileNotFoundError(f"Missing metadata file: {csv_path}")
        frames[split] = load_metadata(csv_path, root)
        validate_file_existence(frames[split], split, check_all=True)

    audit_split_leakage(frames)

    run_root = Path("runs") / CFG.experiment_name
    run_root.mkdir(parents=True, exist_ok=True)
    for split, frame in frames.items():
        locked_path = run_root / f"{split}_manifest_locked.csv"
        frame.sort_values("sample_id").to_csv(locked_path, index=False)
        print(f"Locked {split}: {locked_path} | SHA256={sha256_file(locked_path)}")
    return frames

# Run before training:
# SPLIT_FRAMES = prepare_manifests()

## 3 — Audio loading, aligned cropping, and SNR-balanced loaders

In [ ]:
def load_audio_mono(path: str, target_sr: int = CFG.sample_rate) -> np.ndarray:
    try:
        wav, sr = sf.read(path, dtype="float32")
        if wav.ndim == 2:
            wav = wav.mean(axis=1)
        if sr != target_sr:
            wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=target_sr)
        return np.asarray(wav, dtype=np.float32)
    except Exception:
        wav, _ = librosa.load(path, sr=target_sr, mono=True)
        return np.asarray(wav, dtype=np.float32)


def aligned_crop_or_pad(noisy: np.ndarray, clean: np.ndarray, crop_length: int, augment: bool):
    max_len = max(len(noisy), len(clean))
    noisy = np.pad(noisy, (0, max_len - len(noisy)))
    clean = np.pad(clean, (0, max_len - len(clean)))

    if max_len >= crop_length:
        start = np.random.randint(0, max_len - crop_length + 1) if augment else (max_len - crop_length) // 2
        noisy = noisy[start:start + crop_length]
        clean = clean[start:start + crop_length]
    else:
        pad = crop_length - max_len
        noisy = np.pad(noisy, (0, pad))
        clean = np.pad(clean, (0, pad))
    return noisy.astype(np.float32), clean.astype(np.float32)


class SpeechEnhancementDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, crop_length: int = CFG.crop_length, augment: bool = False):
        self.frame = frame.reset_index(drop=True).copy()
        self.crop_length = crop_length
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index: int):
        row = self.frame.iloc[index]
        noisy = load_audio_mono(row["noisy_path"])
        clean = load_audio_mono(row["clean_path"])
        noisy, clean = aligned_crop_or_pad(noisy, clean, self.crop_length, self.augment)

        if self.augment:
            gain = np.random.uniform(0.80, 1.20)
            polarity = -1.0 if np.random.rand() < 0.5 else 1.0
            noisy = noisy * gain * polarity
            clean = clean * gain * polarity

        peak = max(np.max(np.abs(noisy)), np.max(np.abs(clean)), 1.0)
        noisy = np.clip(noisy / peak, -1.0, 1.0)
        clean = np.clip(clean / peak, -1.0, 1.0)

        return {
            "noisy": torch.from_numpy(noisy),
            "clean": torch.from_numpy(clean),
            "snr_db": torch.tensor(float(row["snr_db"]) if pd.notna(row["snr_db"]) else 0.0),
            "sample_id": row["sample_id"],
        }


def make_snr_balanced_sampler(frame: pd.DataFrame, seed: int):
    counts = frame["snr_db"].value_counts(dropna=False)
    weights = frame["snr_db"].map(lambda s: 1.0 / counts.loc[s]).to_numpy(dtype=np.float64)
    generator = torch.Generator().manual_seed(seed)
    return WeightedRandomSampler(weights, num_samples=len(frame), replacement=True, generator=generator)


def make_dataloaders(split_frames: dict[str, pd.DataFrame], seed: int):
    train_ds = SpeechEnhancementDataset(split_frames["train"], augment=True)
    val_ds = SpeechEnhancementDataset(split_frames["val"], augment=False)
    generator = torch.Generator().manual_seed(seed)

    common = dict(
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(CFG.num_workers > 0),
        worker_init_fn=seed_worker,
        generator=generator,
    )
    if CFG.num_workers > 0:
        common["prefetch_factor"] = 2

    if CFG.balance_snr and split_frames["train"]["snr_db"].notna().any():
        train_loader = DataLoader(train_ds, sampler=make_snr_balanced_sampler(split_frames["train"], seed), **common)
    else:
        train_loader = DataLoader(train_ds, shuffle=True, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)

    print(f"Train samples: {len(train_ds):,} | Validation samples: {len(val_ds):,}")
    return train_loader, val_loader

## 4 — STFT frontend and reconstruction

In [ ]:
class STFTProcessor(nn.Module):
    def __init__(self, n_fft=CFG.n_fft, hop_length=CFG.hop_length, win_length=CFG.win_length):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", torch.hann_window(win_length), persistent=False)

    def forward(self, waveform: torch.Tensor) -> torch.Tensor:
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        spectrum = torch.stft(
            waveform.float(),
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window.float(),
            center=True,
            pad_mode="reflect",
            return_complex=True,
        )
        return torch.stack([spectrum.real, spectrum.imag], dim=1).permute(0, 1, 3, 2).contiguous()

    def inverse(self, spectrum_ri: torch.Tensor, length: int | None = None) -> torch.Tensor:
        real = spectrum_ri[:, 0].permute(0, 2, 1).contiguous().float()
        imag = spectrum_ri[:, 1].permute(0, 2, 1).contiguous().float()
        spectrum = torch.complex(real, imag)
        return torch.istft(
            spectrum,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.win_length,
            window=self.window.float(),
            center=True,
            length=length,
        )


def ri_to_features(spectrum_ri: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    real = spectrum_ri[:, 0:1]
    imag = spectrum_ri[:, 1:2]
    magnitude = torch.sqrt(real.square() + imag.square() + eps)
    scale = magnitude.detach().amax(dim=(-2, -1), keepdim=True).clamp_min(eps)
    return torch.cat([
        real / scale,
        imag / scale,
        torch.log1p(magnitude) / torch.log1p(scale),
        imag / magnitude.clamp_min(eps),   # sin phase
        real / magnitude.clamp_min(eps),   # cos phase
    ], dim=1)


def match_tf(left: torch.Tensor, right: torch.Tensor):
    time_bins = min(left.shape[-2], right.shape[-2])
    freq_bins = min(left.shape[-1], right.shape[-1])
    return left[..., :time_bins, :freq_bins], right[..., :time_bins, :freq_bins]

## 5 — Exact differentiable quantum state-vector simulator

For each audio segment, the conditioner compresses global channel statistics into six angles. The QNN then performs:

- angle encoding with `RY` gates;
- trainable `RZ-RY-RZ` rotations;
- ring CNOT entanglement;
- data re-uploading in each variational layer;
- measurements of six `⟨Zᵢ⟩` values and six neighbouring `⟨ZᵢZᵢ₊₁⟩` correlations.

The state has only `2⁶ = 64` complex amplitudes. The simulator uses actual unitary matrices and Born-rule expectation values; it is not a sine/cosine feature proxy.

In [ ]:
class ExactStatevectorVQC(nn.Module):
    """Batched, differentiable, exact pure-state simulator implemented in PyTorch."""
    def __init__(self, n_qubits: int, n_layers: int, entanglement: str = "ring", data_reuploading: bool = True):
        super().__init__()
        if entanglement not in {"ring", "linear", "none"}:
            raise ValueError("entanglement must be 'ring', 'linear', or 'none'")
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.entanglement = entanglement
        self.data_reuploading = data_reuploading
        self.theta = nn.Parameter(0.05 * torch.randn(n_layers, n_qubits, 3))

        dimension = 2 ** n_qubits
        z_signs = []
        for wire in range(n_qubits):
            shift = n_qubits - 1 - wire
            z_signs.append([1.0 if ((basis >> shift) & 1) == 0 else -1.0 for basis in range(dimension)])
        z_signs = torch.tensor(z_signs, dtype=torch.float32)
        zz_signs = z_signs * torch.roll(z_signs, shifts=-1, dims=0)
        self.register_buffer("z_signs", z_signs, persistent=False)
        self.register_buffer("zz_signs", zz_signs, persistent=False)

        edges = []
        if entanglement == "ring":
            edges = [(wire, (wire + 1) % n_qubits) for wire in range(n_qubits)]
        elif entanglement == "linear":
            edges = [(wire, wire + 1) for wire in range(n_qubits - 1)]
        self.edges = edges

        permutations = []
        for control, target in edges:
            permutation = []
            for basis in range(dimension):
                control_bit = (basis >> (n_qubits - 1 - control)) & 1
                mapped = basis ^ (1 << (n_qubits - 1 - target)) if control_bit else basis
                permutation.append(mapped)
            permutations.append(permutation)
        if permutations:
            permutations = torch.tensor(permutations, dtype=torch.long)
        else:
            permutations = torch.empty((0, dimension), dtype=torch.long)
        self.register_buffer("cnot_permutations", permutations, persistent=False)

        self.runtime_shots: int | None = None
        self.runtime_readout_error: float = 0.0
        self.runtime_angle_noise_std: float = 0.0

    @property
    def output_dim(self) -> int:
        return 2 * self.n_qubits

    def set_runtime_noise(self, shots: int | None = None, readout_error: float = 0.0, angle_noise_std: float = 0.0):
        if shots is not None and shots <= 0:
            raise ValueError("shots must be positive or None")
        if not 0.0 <= readout_error < 0.5:
            raise ValueError("readout_error must be in [0, 0.5)")
        if angle_noise_std < 0:
            raise ValueError("angle_noise_std must be non-negative")
        self.runtime_shots = shots
        self.runtime_readout_error = float(readout_error)
        self.runtime_angle_noise_std = float(angle_noise_std)

    def _apply_single_qubit(self, state: torch.Tensor, gate: torch.Tensor, wire: int) -> torch.Tensor:
        batch = state.shape[0]
        axis = wire + 1
        tensor = state.reshape(batch, *([2] * self.n_qubits))
        permutation = [0] + [a for a in range(1, self.n_qubits + 1) if a != axis] + [axis]
        inverse = [0] * len(permutation)
        for index, value in enumerate(permutation):
            inverse[value] = index
        tensor = tensor.permute(permutation).reshape(batch, -1, 2)
        # Avoid einsum here: on CUDA, float32_matmul_precision='high' may use
        # reduced-precision complex matrix multiplication and slowly violate
        # state normalization. Explicit 2x2 arithmetic preserves complex64
        # accuracy while leaving TF32 available to the classical backbone.
        amplitude_zero = tensor[..., 0]
        amplitude_one = tensor[..., 1]
        if gate.ndim == 2:
            output_zero = amplitude_zero * gate[0, 0] + amplitude_one * gate[0, 1]
            output_one = amplitude_zero * gate[1, 0] + amplitude_one * gate[1, 1]
        else:
            output_zero = amplitude_zero * gate[:, None, 0, 0] + amplitude_one * gate[:, None, 0, 1]
            output_one = amplitude_zero * gate[:, None, 1, 0] + amplitude_one * gate[:, None, 1, 1]
        tensor = torch.stack([output_zero, output_one], dim=-1)
        return tensor.reshape(batch, *([2] * self.n_qubits)).permute(inverse).reshape(batch, -1)

    @staticmethod
    def _ry(angle: torch.Tensor) -> torch.Tensor:
        cosine = torch.cos(angle / 2)
        sine = torch.sin(angle / 2)
        gate = torch.stack([
            torch.stack([cosine, -sine], dim=-1),
            torch.stack([sine, cosine], dim=-1),
        ], dim=-2)
        return gate.to(torch.complex64)

    @staticmethod
    def _rz(angle: torch.Tensor) -> torch.Tensor:
        angle_c = angle.to(torch.complex64)
        phase_zero = torch.exp(-0.5j * angle_c)
        phase_one = torch.exp(0.5j * angle_c)
        zero = torch.zeros_like(phase_zero)
        return torch.stack([
            torch.stack([phase_zero, zero], dim=-1),
            torch.stack([zero, phase_one], dim=-1),
        ], dim=-2)

    def simulate_state(self, inputs: torch.Tensor, angle_noise_std: float | None = None) -> torch.Tensor:
        inputs = inputs.float()
        batch = inputs.shape[0]
        dimension = 2 ** self.n_qubits
        state = torch.zeros(batch, dimension, dtype=torch.complex64, device=inputs.device)
        state[:, 0] = 1.0 + 0.0j
        noise_std = self.runtime_angle_noise_std if angle_noise_std is None else angle_noise_std

        for layer in range(self.n_layers):
            if self.data_reuploading or layer == 0:
                for wire in range(self.n_qubits):
                    angle = inputs[:, wire]
                    if noise_std > 0:
                        angle = angle + noise_std * torch.randn_like(angle)
                    state = self._apply_single_qubit(state, self._ry(angle), wire)

            for wire in range(self.n_qubits):
                parameters = self.theta[layer, wire]
                if noise_std > 0:
                    parameters = parameters + noise_std * torch.randn_like(parameters)
                state = self._apply_single_qubit(state, self._rz(parameters[0].expand(batch)), wire)
                state = self._apply_single_qubit(state, self._ry(parameters[1].expand(batch)), wire)
                state = self._apply_single_qubit(state, self._rz(parameters[2].expand(batch)), wire)

            for edge_index in range(len(self.edges)):
                # CNOT is self-inverse, so this basis permutation is valid in gather form.
                state = state[:, self.cnot_permutations[edge_index]]

        return state

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        state = self.simulate_state(inputs)
        probabilities = state.abs().square().real
        z = probabilities @ self.z_signs.t().to(probabilities)
        zz = probabilities @ self.zz_signs.t().to(probabilities)
        expectations = torch.cat([z, zz], dim=-1).clamp(-1.0, 1.0)

        readout_error = self.runtime_readout_error
        shots = self.runtime_shots
        if readout_error > 0:
            expectations = (1.0 - 2.0 * readout_error) * expectations
        if shots is not None:
            probability_plus = ((expectations + 1.0) / 2.0).clamp(0.0, 1.0)
            counts = torch.distributions.Binomial(
                total_count=float(shots), probs=probability_plus
            ).sample()
            expectations = 2.0 * counts / float(shots) - 1.0
        return expectations


class ParameterMatchedMLPCore(nn.Module):
    """
    Classical MLP control with exactly the same core trainable-parameter count as the VQC.

    For the main configuration: VQC = 2 layers × 6 qubits × 3 angles = 36 parameters.
    MLP = Linear(6, 3, no bias) + Linear(3, 6, no bias) = 36 parameters.
    """
    def __init__(self, n_qubits: int, q_layers: int):
        super().__init__()
        target_parameters = q_layers * n_qubits * 3
        denominator = 2 * n_qubits
        if target_parameters % denominator != 0:
            raise ValueError(
                "Exact MLP matching requires q_layers*3 to be even. Use q_layers=2 or 4 for this notebook."
            )
        hidden = target_parameters // denominator
        self.net = nn.Sequential(
            nn.Linear(n_qubits, hidden, bias=False),
            nn.Tanh(),
            nn.Linear(hidden, n_qubits, bias=False),
            nn.Tanh(),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        values = self.net(inputs)
        neighbour_products = values * torch.roll(values, shifts=-1, dims=-1)
        return torch.cat([values, neighbour_products], dim=-1)

### Quantum simulator unit tests

In [ ]:
def run_quantum_unit_tests():
    seed_everything(123)
    circuit = ExactStatevectorVQC(
        n_qubits=CFG.n_qubits,
        n_layers=CFG.q_layers,
        entanglement=CFG.entanglement,
        data_reuploading=CFG.data_reuploading,
    ).to(DEVICE)
    inputs = torch.randn(4, CFG.n_qubits, device=DEVICE, requires_grad=True)
    state = circuit.simulate_state(inputs)
    norms = state.abs().square().sum(dim=-1)
    assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5), norms

    outputs = circuit(inputs)
    assert outputs.shape == (4, 2 * CFG.n_qubits)
    assert outputs.min() >= -1.0001 and outputs.max() <= 1.0001

    loss = outputs.square().mean()
    loss.backward()
    assert circuit.theta.grad is not None and torch.isfinite(circuit.theta.grad).all()
    assert inputs.grad is not None and torch.isfinite(inputs.grad).all()

    # Bell-state check: RY(pi/2) on q0 followed by CNOT(0->1) gives |00>+|11>.
    bell = ExactStatevectorVQC(2, 1, entanglement="linear", data_reuploading=False).to(DEVICE)
    test_state = torch.zeros(1, 4, dtype=torch.complex64, device=DEVICE)
    test_state[:, 0] = 1.0
    test_state = bell._apply_single_qubit(test_state, bell._ry(torch.tensor([math.pi / 2], device=DEVICE)), 0)
    test_state = test_state[:, bell.cnot_permutations[0]]
    bell_probs = test_state.abs().square().squeeze(0)
    expected = torch.tensor([0.5, 0.0, 0.0, 0.5], device=DEVICE)
    assert torch.allclose(bell_probs, expected, atol=1e-5), bell_probs

    print("Quantum tests passed.")
    print("State dimension:", 2 ** CFG.n_qubits)
    print("VQC output shape:", tuple(outputs.shape))
    print("Mean |quantum gradient|:", circuit.theta.grad.abs().mean().item())

run_quantum_unit_tests()

### Optional independent verification against PennyLane

In [ ]:
def verify_statevector_against_pennylane(atol: float = 2e-5):
    try:
        import pennylane as qml
    except ImportError:
        print("PennyLane is not installed. Run the optional installation cell to perform this check.")
        return None

    seed_everything(7)
    custom = ExactStatevectorVQC(
        CFG.n_qubits, CFG.q_layers, CFG.entanglement, CFG.data_reuploading
    ).cpu()
    x = torch.randn(CFG.n_qubits)
    theta = custom.theta.detach().clone()
    dev = qml.device("default.qubit", wires=CFG.n_qubits, shots=None)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def reference(inputs, weights):
        for layer in range(CFG.q_layers):
            if CFG.data_reuploading or layer == 0:
                for wire in range(CFG.n_qubits):
                    qml.RY(inputs[wire], wires=wire)
            for wire in range(CFG.n_qubits):
                qml.RZ(weights[layer, wire, 0], wires=wire)
                qml.RY(weights[layer, wire, 1], wires=wire)
                qml.RZ(weights[layer, wire, 2], wires=wire)
            if CFG.entanglement == "ring":
                for wire in range(CFG.n_qubits):
                    qml.CNOT(wires=[wire, (wire + 1) % CFG.n_qubits])
            elif CFG.entanglement == "linear":
                for wire in range(CFG.n_qubits - 1):
                    qml.CNOT(wires=[wire, wire + 1])
        observables = [qml.PauliZ(wire) for wire in range(CFG.n_qubits)]
        observables += [
            qml.PauliZ(wire) @ qml.PauliZ((wire + 1) % CFG.n_qubits)
            for wire in range(CFG.n_qubits)
        ]
        return [qml.expval(obs) for obs in observables]

    custom_output = custom(x.unsqueeze(0)).squeeze(0).detach()
    reference_output = torch.stack(reference(x, theta)).float().detach()
    max_error = (custom_output - reference_output).abs().max().item()
    print("Maximum custom-vs-PennyLane error:", max_error)
    assert max_error < atol, (custom_output, reference_output)
    print("Independent PennyLane verification passed.")
    return max_error

# verify_statevector_against_pennylane()

## 6 — Shared lightweight backbone and parameter-matched conditioners

In [ ]:
class DepthwiseConvGLU(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3, dropout: float = CFG.dropout):
        super().__init__()
        padding = kernel_size // 2
        self.depthwise = nn.Conv2d(
            channels, channels, kernel_size, padding=padding, groups=channels, bias=False
        )
        self.pointwise = nn.Conv2d(channels, 2 * channels, kernel_size=1)
        self.norm = nn.GroupNorm(1, channels)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        value, gate = self.pointwise(self.depthwise(inputs)).chunk(2, dim=1)
        update = value * torch.sigmoid(gate)
        return inputs + self.dropout(self.norm(update))


class MicroEnhancementBlock(nn.Module):
    def __init__(self, channels: int, dropout: float = CFG.dropout):
        super().__init__()
        self.norm1 = nn.GroupNorm(1, channels)
        self.temporal = nn.Conv2d(
            channels, channels, kernel_size=(5, 1), padding=(2, 0), groups=channels, bias=False
        )
        self.frequency = nn.Conv2d(
            channels, channels, kernel_size=(1, 5), padding=(0, 2), groups=channels, bias=False
        )
        self.glu = nn.Conv2d(channels, 2 * channels, kernel_size=1)
        self.dropout = nn.Dropout2d(dropout)
        self.norm2 = nn.GroupNorm(1, channels)
        self.ffn = nn.Sequential(
            nn.Conv2d(channels, 2 * channels, kernel_size=1),
            nn.SiLU(),
            nn.Dropout2d(dropout),
            nn.Conv2d(2 * channels, channels, kernel_size=1),
            nn.Dropout2d(dropout),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.norm1(inputs)
        hidden = self.temporal(hidden) + self.frequency(hidden)
        value, gate = self.glu(hidden).chunk(2, dim=1)
        outputs = inputs + self.dropout(value * torch.sigmoid(gate))
        return outputs + self.ffn(self.norm2(outputs))


class GlobalChannelConditioner(nn.Module):
    """One global conditioner invocation per audio segment."""
    def __init__(self, channels: int, variant: str):
        super().__init__()
        if variant not in {"baseline", "mlp", "qnn"}:
            raise ValueError("variant must be baseline, mlp, or qnn")
        self.variant = variant
        self.channels = channels
        if variant == "baseline":
            return

        self.input_projection = nn.Sequential(
            nn.Linear(2 * channels, CFG.n_qubits),
            nn.LayerNorm(CFG.n_qubits),
        )
        if variant == "qnn":
            self.core = ExactStatevectorVQC(
                CFG.n_qubits, CFG.q_layers, CFG.entanglement, CFG.data_reuploading
            )
        else:
            self.core = ParameterMatchedMLPCore(CFG.n_qubits, CFG.q_layers)

        # Both variants receive 2*n_qubits features and predict channel-wise scale and bias.
        self.output_projection = nn.Linear(2 * CFG.n_qubits, 2 * channels)
        # Small initial modulation protects the pretrained backbone.
        self.logit_strength = nn.Parameter(torch.tensor(-2.2))

    def set_quantum_runtime(self, shots=None, readout_error=0.0, angle_noise_std=0.0):
        if self.variant == "qnn":
            self.core.set_runtime_noise(shots, readout_error, angle_noise_std)

    def forward(self, inputs: torch.Tensor, return_aux: bool = False):
        if self.variant == "baseline":
            return (inputs, {}) if return_aux else inputs

        means = inputs.mean(dim=(-2, -1))
        standard_deviations = inputs.std(dim=(-2, -1), unbiased=False)
        statistics = torch.cat([means, standard_deviations], dim=-1)
        angles = math.pi * torch.tanh(self.input_projection(statistics))

        if self.variant == "qnn":
            # Complex state-vector operations remain float32/complex64 under AMP.
            with torch.autocast(device_type=inputs.device.type, enabled=False):
                latent = self.core(angles.float())
        else:
            latent = self.core(angles)

        scale, bias = self.output_projection(latent).chunk(2, dim=-1)
        scale = torch.tanh(scale).unsqueeze(-1).unsqueeze(-1)
        bias = torch.tanh(bias).unsqueeze(-1).unsqueeze(-1)
        strength = 0.25 * torch.sigmoid(self.logit_strength)
        outputs = inputs * (1.0 + strength * scale) + strength * bias

        if return_aux:
            return outputs, {
                "angles": angles.detach(),
                "latent": latent.detach(),
                "scale": scale.detach(),
                "bias": bias.detach(),
                "strength": strength.detach(),
            }
        return outputs


class QMicroSESVQC(nn.Module):
    def __init__(self, variant: str):
        super().__init__()
        self.variant = variant
        self.stft = STFTProcessor()
        channels = CFG.channels
        self.input_projection = nn.Sequential(
            nn.Conv2d(5, channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, channels),
            nn.SiLU(),
            nn.Dropout2d(CFG.dropout),
        )
        self.pre = DepthwiseConvGLU(channels)
        self.blocks = nn.ModuleList([MicroEnhancementBlock(channels) for _ in range(CFG.num_blocks)])
        self.conditioner = GlobalChannelConditioner(channels, variant)
        self.post = DepthwiseConvGLU(channels)
        self.head = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.SiLU(),
            nn.Dropout2d(CFG.dropout),
            nn.Conv2d(channels, 4, kernel_size=1),
        )

    def set_quantum_runtime(self, shots=None, readout_error=0.0, angle_noise_std=0.0):
        self.conditioner.set_quantum_runtime(shots, readout_error, angle_noise_std)

    def forward(self, noisy_waveform: torch.Tensor, return_aux: bool = False):
        waveform_length = noisy_waveform.shape[-1]
        noisy_stft = self.stft(noisy_waveform.float())
        features = ri_to_features(noisy_stft)

        hidden = self.pre(self.input_projection(features))
        conditioner_aux = {}
        for block_index, block in enumerate(self.blocks, start=1):
            hidden = block(hidden)
            if block_index == CFG.conditioner_after_block:
                if return_aux:
                    hidden, conditioner_aux = self.conditioner(hidden, return_aux=True)
                else:
                    hidden = self.conditioner(hidden)

        outputs = self.head(self.post(hidden))
        outputs, noisy_stft = match_tf(outputs, noisy_stft)

        magnitude_mask = 2.0 * torch.sigmoid(outputs[:, 0:1])
        phase_residual = CFG.phase_residual_scale * torch.tanh(outputs[:, 1:2])
        real_residual = CFG.complex_residual_scale * torch.tanh(outputs[:, 2:3])
        imag_residual = CFG.complex_residual_scale * torch.tanh(outputs[:, 3:4])

        noisy_real = noisy_stft[:, 0:1]
        noisy_imag = noisy_stft[:, 1:2]
        noisy_magnitude = torch.sqrt(noisy_real.square() + noisy_imag.square() + 1e-8)
        noisy_phase = torch.atan2(noisy_imag, noisy_real)

        enhanced_magnitude = noisy_magnitude * magnitude_mask
        enhanced_phase = noisy_phase + phase_residual
        enhanced_real = enhanced_magnitude * torch.cos(enhanced_phase) + real_residual * noisy_magnitude
        enhanced_imag = enhanced_magnitude * torch.sin(enhanced_phase) + imag_residual * noisy_magnitude
        enhanced_stft = torch.cat([enhanced_real, enhanced_imag], dim=1)
        enhanced_waveform = self.stft.inverse(enhanced_stft, length=waveform_length)

        if return_aux:
            return enhanced_waveform, enhanced_stft, {
                "magnitude_mask": magnitude_mask.detach(),
                "phase_residual": phase_residual.detach(),
                **conditioner_aux,
            }
        return enhanced_waveform, enhanced_stft

    @torch.no_grad()
    def enhance(self, noisy_waveform: torch.Tensor):
        self.eval()
        return self.forward(noisy_waveform)[0]

## 7 — Parameter matching and quantum resource report

In [ ]:
def parameter_count(module: nn.Module) -> int:
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


def conditioner_core_count(model: QMicroSESVQC) -> int:
    if model.variant == "baseline":
        return 0
    return parameter_count(model.conditioner.core)


def quantum_resource_report():
    encoding_gates = CFG.n_qubits * CFG.q_layers if CFG.data_reuploading else CFG.n_qubits
    trainable_single_qubit_gates = CFG.q_layers * CFG.n_qubits * 3
    cnot_per_layer = CFG.n_qubits if CFG.entanglement == "ring" else max(CFG.n_qubits - 1, 0)
    cnot_gates = CFG.q_layers * cnot_per_layer if CFG.entanglement != "none" else 0
    return {
        "qubits": CFG.n_qubits,
        "statevector_dimension": 2 ** CFG.n_qubits,
        "variational_layers": CFG.q_layers,
        "trainable_quantum_parameters": CFG.q_layers * CFG.n_qubits * 3,
        "encoding_RY_gates_per_segment": encoding_gates,
        "trainable_rotation_gates_per_segment": trainable_single_qubit_gates,
        "CNOT_gates_per_segment": cnot_gates,
        "total_gates_per_segment": encoding_gates + trainable_single_qubit_gates + cnot_gates,
        "measured_observables": 2 * CFG.n_qubits,
        "circuit_invocations_per_segment": 1,
    }

rows = []
models_for_count = {}
for variant in CFG.variants:
    model = QMicroSESVQC(variant)
    models_for_count[variant] = model
    rows.append({
        "variant": variant,
        "total_parameters": parameter_count(model),
        "conditioner_parameters": parameter_count(model.conditioner),
        "core_parameters": conditioner_core_count(model),
    })
parameter_table = pd.DataFrame(rows)
display(parameter_table)

mlp_params = parameter_table.loc[parameter_table.variant == "mlp", "total_parameters"].item()
qnn_params = parameter_table.loc[parameter_table.variant == "qnn", "total_parameters"].item()
mlp_core = parameter_table.loc[parameter_table.variant == "mlp", "core_parameters"].item()
qnn_core = parameter_table.loc[parameter_table.variant == "qnn", "core_parameters"].item()
assert mlp_params == qnn_params, (mlp_params, qnn_params)
assert mlp_core == qnn_core, (mlp_core, qnn_core)
print("MLP and QNN are exactly parameter matched.")
display(pd.DataFrame([quantum_resource_report()]))
del models_for_count
gc.collect()

### Model forward/backward smoke test

In [ ]:
def model_smoke_test():
    seed_everything(99)
    # Short waveform is enough to validate tensor shapes and gradients.
    noisy = torch.randn(2, CFG.sample_rate, device=DEVICE) * 0.02
    for variant in CFG.variants:
        model = QMicroSESVQC(variant).to(DEVICE)
        enhanced, spectrum = model(noisy)
        assert enhanced.shape == noisy.shape
        assert spectrum.shape[1] == 2
        loss = enhanced.square().mean() + spectrum.square().mean()
        loss.backward()
        gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        assert gradients and all(torch.isfinite(g).all() for g in gradients)
        print(
            f"{variant:>8}: params={parameter_count(model):,}, "
            f"enhanced={tuple(enhanced.shape)}, finite_gradients=True"
        )
        del model, enhanced, spectrum, loss
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

model_smoke_test()

## 8 — Multi-objective speech-enhancement loss

In [ ]:
def si_sdr_per_sample(estimate: torch.Tensor, reference: torch.Tensor, eps: float = 1e-8):
    estimate = estimate - estimate.mean(dim=-1, keepdim=True)
    reference = reference - reference.mean(dim=-1, keepdim=True)
    projection_scale = (estimate * reference).sum(dim=-1, keepdim=True) / (
        reference.square().sum(dim=-1, keepdim=True) + eps
    )
    target = projection_scale * reference
    residual = estimate - target
    return 10.0 * torch.log10(
        (target.square().sum(dim=-1) + eps) / (residual.square().sum(dim=-1) + eps)
    )


class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, resolutions=((256, 64, 256), (512, 128, 512), (1024, 256, 1024))):
        super().__init__()
        self.resolutions = resolutions
        for index, (_, _, win_length) in enumerate(resolutions):
            self.register_buffer(f"window_{index}", torch.hann_window(win_length), persistent=False)

    def forward(self, estimate: torch.Tensor, reference: torch.Tensor):
        losses = []
        for index, (n_fft, hop, win_length) in enumerate(self.resolutions):
            window = getattr(self, f"window_{index}").float()
            estimate_stft = torch.stft(
                estimate.float(), n_fft=n_fft, hop_length=hop, win_length=win_length,
                window=window, return_complex=True, center=True
            )
            reference_stft = torch.stft(
                reference.float(), n_fft=n_fft, hop_length=hop, win_length=win_length,
                window=window, return_complex=True, center=True
            )
            estimate_mag = estimate_stft.abs().clamp_min(1e-7)
            reference_mag = reference_stft.abs().clamp_min(1e-7)
            spectral_convergence = torch.linalg.vector_norm(reference_mag - estimate_mag, dim=(-2, -1)) / (
                torch.linalg.vector_norm(reference_mag, dim=(-2, -1)) + 1e-7
            )
            log_magnitude = F.l1_loss(torch.log(estimate_mag), torch.log(reference_mag), reduction="none").mean(dim=(-2, -1))
            losses.append(spectral_convergence + log_magnitude)
        return torch.stack(losses, dim=0).mean()


class ResearchSpeechLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.huber = nn.SmoothL1Loss(beta=1.0)
        self.mrstft = MultiResolutionSTFTLoss()

    def forward(self, enhanced_waveform, enhanced_stft, clean_waveform, clean_stft):
        enhanced_stft, clean_stft = match_tf(enhanced_stft, clean_stft)
        enhanced_real, enhanced_imag = enhanced_stft[:, 0], enhanced_stft[:, 1]
        clean_real, clean_imag = clean_stft[:, 0], clean_stft[:, 1]
        enhanced_mag = torch.sqrt(enhanced_real.square() + enhanced_imag.square() + 1e-8)
        clean_mag = torch.sqrt(clean_real.square() + clean_imag.square() + 1e-8)

        loss_complex = self.huber(enhanced_stft, clean_stft)
        loss_magnitude = self.huber(torch.log1p(enhanced_mag), torch.log1p(clean_mag))

        enhanced_phase = torch.atan2(enhanced_imag, enhanced_real)
        clean_phase = torch.atan2(clean_imag, clean_real)
        phase_weight = (clean_mag / (clean_mag.mean(dim=(-2, -1), keepdim=True) + 1e-8)).clamp(0.0, 2.0)
        loss_phase = (phase_weight * (1.0 - torch.cos(enhanced_phase - clean_phase))).mean()

        sisdr_db = si_sdr_per_sample(enhanced_waveform, clean_waveform).mean()
        loss_sisdr = -sisdr_db / 20.0
        loss_mrstft = self.mrstft(enhanced_waveform, clean_waveform)

        total = (
            0.30 * loss_complex
            + 0.25 * loss_magnitude
            + 0.05 * loss_phase
            + 0.20 * loss_mrstft
            + 0.20 * loss_sisdr
        )
        return {
            "total": total,
            "complex": loss_complex.detach(),
            "magnitude": loss_magnitude.detach(),
            "phase": loss_phase.detach(),
            "mrstft": loss_mrstft.detach(),
            "sisdr_db": sisdr_db.detach(),
        }

## 9 — Training utilities and common-initialisation protocol

In [ ]:
def amp_context():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


class EarlyStopping:
    def __init__(self, patience: int, min_delta: float):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float("inf")
        self.bad_epochs = 0

    def update(self, value: float) -> bool:
        improved = value < self.best - self.min_delta
        if improved:
            self.best = value
            self.bad_epochs = 0
        else:
            self.bad_epochs += 1
        return improved

    @property
    def should_stop(self):
        return self.bad_epochs >= self.patience


def build_optimizer(model: QMicroSESVQC):
    conditioner_parameters = []
    backbone_parameters = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        if name.startswith("conditioner."):
            conditioner_parameters.append(parameter)
        else:
            backbone_parameters.append(parameter)

    groups = [{"params": backbone_parameters, "lr": CFG.backbone_lr, "name": "backbone"}]
    if conditioner_parameters:
        groups.append({"params": conditioner_parameters, "lr": CFG.conditioner_lr, "name": "conditioner"})

    optimizer = torch.optim.AdamW(groups, weight_decay=CFG.weight_decay, betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
    )
    return optimizer, scheduler


def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    totals = {"total": 0.0, "complex": 0.0, "magnitude": 0.0, "phase": 0.0, "mrstft": 0.0, "sisdr_db": 0.0}
    sample_count = 0
    progress = tqdm(loader, desc="Train" if training else "Validation", leave=False)

    for batch in progress:
        noisy = batch["noisy"].to(DEVICE, non_blocking=True)
        clean = batch["clean"].to(DEVICE, non_blocking=True)
        batch_size = noisy.shape[0]

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            clean_stft = model.stft(clean.float())

        grad_context = torch.enable_grad() if training else torch.no_grad()
        with grad_context:
            with amp_context():
                enhanced, enhanced_stft = model(noisy)
                losses = criterion(enhanced, enhanced_stft, clean, clean_stft)
                loss = losses["total"]

            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()

        for key in totals:
            totals[key] += float(losses[key].detach().cpu()) * batch_size
        sample_count += batch_size
        progress.set_postfix({
            "loss": f"{totals['total']/sample_count:.4f}",
            "SI-SDR": f"{totals['sisdr_db']/sample_count:.2f}",
        })

    return {key: value / max(sample_count, 1) for key, value in totals.items()}


def load_shared_backbone(model: QMicroSESVQC, checkpoint_path: str | Path):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state = checkpoint["model_state"]
    incompatible = model.load_state_dict(state, strict=False)
    unexpected = [key for key in incompatible.unexpected_keys if not key.startswith("conditioner.")]
    missing = [key for key in incompatible.missing_keys if not key.startswith("conditioner.")]
    if unexpected or missing:
        raise RuntimeError(f"Unexpected shared-backbone mismatch. Missing={missing}, unexpected={unexpected}")
    return checkpoint


def fit_model(
    variant: str,
    seed: int,
    split_frames: dict[str, pd.DataFrame],
    stage_name: str,
    max_epochs: int,
    initial_checkpoint: str | Path | None = None,
):
    seed_everything(seed)
    train_loader, val_loader = make_dataloaders(split_frames, seed)
    model = QMicroSESVQC(variant).to(DEVICE)
    if initial_checkpoint is not None:
        loaded = load_shared_backbone(model, initial_checkpoint)
        print(f"Loaded common backbone from epoch {loaded['epoch']}: {initial_checkpoint}")

    run_dir = Path("runs") / CFG.experiment_name / f"seed_{seed}" / stage_name / variant
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / "best.pt"
    history_path = run_dir / "history.csv"

    criterion = ResearchSpeechLoss().to(DEVICE)
    optimizer, scheduler = build_optimizer(model)
    scaler = make_grad_scaler()
    early = EarlyStopping(CFG.early_stopping_patience, CFG.min_delta)
    start_epoch = 1
    history = []

    if CFG.resume and checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state"])
        start_epoch = checkpoint["epoch"] + 1
        early.best = checkpoint["val_total"]
        if history_path.exists():
            history = pd.read_csv(history_path).to_dict("records")
        print(f"Resuming {variant} from epoch {checkpoint['epoch']}")

    print(
        f"\n{stage_name} | seed={seed} | variant={variant} | "
        f"parameters={parameter_count(model):,}"
    )
    for epoch in range(start_epoch, max_epochs + 1):
        start = time.perf_counter()
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, scaler)
        val_metrics = run_epoch(model, val_loader, criterion)
        scheduler.step(val_metrics["total"])
        elapsed = time.perf_counter() - start

        row = {
            "epoch": epoch,
            "seed": seed,
            "variant": variant,
            "stage": stage_name,
            "seconds": elapsed,
            "lr_backbone": optimizer.param_groups[0]["lr"],
            **{f"train_{key}": value for key, value in train_metrics.items()},
            **{f"val_{key}": value for key, value in val_metrics.items()},
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        improved = early.update(val_metrics["total"])
        marker = "*" if improved else " "
        print(
            f"{marker} epoch={epoch:03d} train={train_metrics['total']:.5f} "
            f"val={val_metrics['total']:.5f} val_SI-SDR={val_metrics['sisdr_db']:.3f} dB "
            f"time={elapsed:.1f}s"
        )
        if improved:
            torch.save({
                "epoch": epoch,
                "seed": seed,
                "variant": variant,
                "stage": stage_name,
                "model_state": model.state_dict(),
                "val_total": val_metrics["total"],
                "val_sisdr_db": val_metrics["sisdr_db"],
                "parameters": parameter_count(model),
                "config": asdict(CFG),
                "quantum_resources": quantum_resource_report() if variant == "qnn" else None,
            }, checkpoint_path)
        if early.should_stop:
            print("Early stopping triggered.")
            break

    best = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best["model_state"])
    return model, pd.DataFrame(history), checkpoint_path


def run_three_arm_protocol(split_frames: dict[str, pd.DataFrame], seeds=CFG.seeds):
    """
    Stage A: pretrain one baseline backbone for each seed.
    Stage B: initialise baseline, MLP, and QNN from that exact checkpoint and fine-tune all arms.
    """
    records = []
    for seed in seeds:
        _, pretrain_history, shared_checkpoint = fit_model(
            variant="baseline",
            seed=seed,
            split_frames=split_frames,
            stage_name="shared_pretrain",
            max_epochs=CFG.shared_pretrain_epochs,
            initial_checkpoint=None,
        )
        for variant in CFG.variants:
            model, history, checkpoint = fit_model(
                variant=variant,
                seed=seed,
                split_frames=split_frames,
                stage_name="matched_finetune",
                max_epochs=CFG.finetune_epochs,
                initial_checkpoint=shared_checkpoint,
            )
            records.append({
                "seed": seed,
                "variant": variant,
                "checkpoint": str(checkpoint),
                "best_val_total": float(history["val_total"].min()),
                "best_val_sisdr_db": float(history["val_sisdr_db"].max()),
                "parameters": parameter_count(model),
            })
            del model
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    summary = pd.DataFrame(records)
    output_path = Path("runs") / CFG.experiment_name / "training_summary.csv"
    summary.to_csv(output_path, index=False)
    display(summary)
    return summary

In [ ]:
# ============================================================
# RESUME-SAFE PATCH
#
# Paste this AFTER the original fit_model() definition.
# No other notebook cells need to be changed.
# ============================================================

def _load_checkpoint_safely(path, map_location=DEVICE):
    """Compatible with different PyTorch torch.load defaults."""
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location,
        )


def _cpu_byte_rng_state(state):
    """Normalize a deserialized RNG state for PyTorch RNG APIs."""
    if isinstance(state, torch.Tensor):
        return state.detach().to(
            device="cpu",
            dtype=torch.uint8,
        ).contiguous()

    return torch.as_tensor(
        state,
        dtype=torch.uint8,
        device="cpu",
    ).contiguous()


def _read_clean_history(history_path):
    """
    Read history.csv and remove duplicated epochs that may have
    been created by the previous resume implementation.
    """
    if not history_path.exists():
        return pd.DataFrame()

    history_df = pd.read_csv(history_path)

    if history_df.empty or "epoch" not in history_df.columns:
        return pd.DataFrame()

    history_df["epoch"] = pd.to_numeric(
        history_df["epoch"],
        errors="coerce",
    )

    history_df = (
        history_df
        .dropna(subset=["epoch"])
        .assign(epoch=lambda frame: frame["epoch"].astype(int))
        .sort_values("epoch")
        .drop_duplicates(subset=["epoch"], keep="last")
        .reset_index(drop=True)
    )

    return history_df


def _reconstruct_early_stopping(history_df):
    """
    Reconstruct early-stopping state from previously recorded
    validation losses.
    """
    early = EarlyStopping(
        CFG.early_stopping_patience,
        CFG.min_delta,
    )

    if (
        not history_df.empty
        and "val_total" in history_df.columns
    ):
        for value in history_df["val_total"]:
            if pd.notna(value):
                early.update(float(value))

    return early


def _atomic_torch_save(state, destination):
    """
    Save through a temporary file so a kernel failure is less
    likely to corrupt the checkpoint.
    """
    destination = Path(destination)
    temporary = destination.with_suffix(
        destination.suffix + ".tmp"
    )

    torch.save(state, temporary)
    os.replace(temporary, destination)


def _atomic_history_save(history_df, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(".csv.tmp")

    history_df.to_csv(temporary, index=False)
    os.replace(temporary, destination)


def fit_model(
    variant: str,
    seed: int,
    split_frames: dict[str, pd.DataFrame],
    stage_name: str,
    max_epochs: int,
    initial_checkpoint: str | Path | None = None,
):
    """
    Resume-safe replacement for the original fit_model().

    Behaviour:
    1. Completed stages are skipped using history.csv.
    2. best.pt remains the model used for final evaluation.
    3. last.pt stores the latest completed training epoch.
    4. Interrupted training resumes from last.pt.
    5. Existing old checkpoints remain compatible.
    """

    seed_everything(seed)

    train_loader, val_loader = make_dataloaders(
        split_frames,
        seed,
    )

    model = QMicroSESVQC(variant).to(DEVICE)

    if initial_checkpoint is not None:
        loaded = load_shared_backbone(
            model,
            initial_checkpoint,
        )

        print(
            f"Loaded common backbone from epoch "
            f"{loaded['epoch']}: {initial_checkpoint}"
        )

    run_dir = (
        Path("runs")
        / CFG.experiment_name
        / f"seed_{seed}"
        / stage_name
        / variant
    )
    run_dir.mkdir(parents=True, exist_ok=True)

    best_path = run_dir / "best.pt"
    last_path = run_dir / "last.pt"
    history_path = run_dir / "history.csv"
    completion_path = run_dir / "complete.json"

    criterion = ResearchSpeechLoss().to(DEVICE)
    optimizer, scheduler = build_optimizer(model)
    scaler = make_grad_scaler()

    # --------------------------------------------------------
    # Read and clean existing history
    # --------------------------------------------------------

    history_df = _read_clean_history(history_path)
    early = _reconstruct_early_stopping(history_df)

    last_history_epoch = (
        int(history_df["epoch"].max())
        if not history_df.empty
        else 0
    )

    reached_max_epochs = (
        last_history_epoch >= max_epochs
    )

    previously_early_stopped = (
        not history_df.empty
        and early.should_stop
    )

    # --------------------------------------------------------
    # Skip an already completed stage
    # --------------------------------------------------------

    if (
        CFG.resume
        and best_path.exists()
        and (
            reached_max_epochs
            or previously_early_stopped
            or completion_path.exists()
        )
    ):
        best_checkpoint = _load_checkpoint_safely(
            best_path,
            DEVICE,
        )

        model.load_state_dict(
            best_checkpoint["model_state"]
        )

        reason = (
            f"history already reached epoch "
            f"{last_history_epoch}/{max_epochs}"
            if reached_max_epochs
            else "early stopping was already completed"
        )

        print(
            f"\nSkipping completed stage:\n"
            f"  seed={seed}\n"
            f"  stage={stage_name}\n"
            f"  variant={variant}\n"
            f"  reason={reason}\n"
            f"  best epoch="
            f"{best_checkpoint['epoch']}"
        )

        completion_data = {
            "seed": seed,
            "stage": stage_name,
            "variant": variant,
            "last_history_epoch": last_history_epoch,
            "best_epoch": int(
                best_checkpoint["epoch"]
            ),
            "reason": (
                "max_epochs"
                if reached_max_epochs
                else "early_stopping"
            ),
        }

        with open(completion_path, "w") as file:
            json.dump(
                completion_data,
                file,
                indent=2,
            )

        return (
            model,
            history_df,
            best_path,
        )

    # --------------------------------------------------------
    # Resume an incomplete stage
    # --------------------------------------------------------

    start_epoch = 1

    if CFG.resume and last_path.exists():
        # Exact continuation for epochs produced after installing
        # this patch.
        checkpoint = _load_checkpoint_safely(
            last_path,
            DEVICE,
        )

        model.load_state_dict(
            checkpoint["model_state"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state"]
        )

        scheduler.load_state_dict(
            checkpoint["scheduler_state"]
        )

        if (
            "scaler_state" in checkpoint
            and checkpoint["scaler_state"] is not None
        ):
            scaler.load_state_dict(
                checkpoint["scaler_state"]
            )

        early.best = checkpoint.get(
            "early_best",
            early.best,
        )

        early.bad_epochs = checkpoint.get(
            "early_bad_epochs",
            early.bad_epochs,
        )

        completed_epoch = int(
            checkpoint["epoch"]
        )

        start_epoch = completed_epoch + 1

        # Prefer the checkpoint's history because it represents
        # exactly the model state stored in last.pt.
        if "history" in checkpoint:
            history_df = pd.DataFrame(
                checkpoint["history"]
            )

        history_df = (
            history_df[
                history_df["epoch"] <= completed_epoch
            ]
            .sort_values("epoch")
            .drop_duplicates("epoch", keep="last")
            .reset_index(drop=True)
        )

        # Restore global random states where available.
        if "python_rng_state" in checkpoint:
            random.setstate(
                checkpoint["python_rng_state"]
            )

        if "numpy_rng_state" in checkpoint:
            np.random.set_state(
                checkpoint["numpy_rng_state"]
            )

        if "torch_rng_state" in checkpoint:
            torch.set_rng_state(
                _cpu_byte_rng_state(
                    checkpoint["torch_rng_state"]
                )
            )

        if (
            torch.cuda.is_available()
            and checkpoint.get(
                "cuda_rng_state"
            ) is not None
        ):
            torch.cuda.set_rng_state_all(
                [
                    _cpu_byte_rng_state(state)
                    for state in checkpoint["cuda_rng_state"]
                ]
            )

        print(
            f"\nExact resume:\n"
            f"  seed={seed}\n"
            f"  stage={stage_name}\n"
            f"  variant={variant}\n"
            f"  last completed epoch="
            f"{completed_epoch}\n"
            f"  next epoch={start_epoch}"
        )

    elif CFG.resume and best_path.exists():
        # Compatibility path for your existing old checkpoints.
        #
        # Old code only stored best.pt. If history went beyond the
        # best epoch, those later weights do not exist. Therefore,
        # training must resume from the best saved epoch once.
        checkpoint = _load_checkpoint_safely(
            best_path,
            DEVICE,
        )

        model.load_state_dict(
            checkpoint["model_state"]
        )

        best_epoch = int(
            checkpoint["epoch"]
        )

        start_epoch = best_epoch + 1
        early.best = float(
            checkpoint["val_total"]
        )
        early.bad_epochs = 0

        # Remove rows after the saved best model. Those rows have
        # metrics but no corresponding recoverable model weights.
        if not history_df.empty:
            history_df = (
                history_df[
                    history_df["epoch"] <= best_epoch
                ]
                .sort_values("epoch")
                .drop_duplicates(
                    "epoch",
                    keep="last",
                )
                .reset_index(drop=True)
            )

            _atomic_history_save(
                history_df,
                history_path,
            )

        print(
            f"\nLegacy checkpoint resume:\n"
            f"  seed={seed}\n"
            f"  stage={stage_name}\n"
            f"  variant={variant}\n"
            f"  best saved epoch={best_epoch}\n"
            f"  next epoch={start_epoch}\n"
            f"  Note: this stage may repeat epochs once because "
            f"the old code did not save last.pt."
        )

    # Convert cleaned history into the original list format.
    history = history_df.to_dict("records")

    print(
        f"\n{stage_name} | "
        f"seed={seed} | "
        f"variant={variant} | "
        f"parameters={parameter_count(model):,}"
    )

    # --------------------------------------------------------
    # Training loop
    # --------------------------------------------------------

    for epoch in range(
        start_epoch,
        max_epochs + 1,
    ):
        epoch_start = time.perf_counter()

        train_metrics = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
        )

        val_metrics = run_epoch(
            model,
            val_loader,
            criterion,
        )

        scheduler.step(
            val_metrics["total"]
        )

        elapsed = (
            time.perf_counter()
            - epoch_start
        )

        row = {
            "epoch": epoch,
            "seed": seed,
            "variant": variant,
            "stage": stage_name,
            "seconds": elapsed,
            "lr_backbone": (
                optimizer
                .param_groups[0]["lr"]
            ),
            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },
            **{
                f"val_{key}": value
                for key, value
                in val_metrics.items()
            },
        }

        # Replace an existing row for this epoch instead of
        # creating duplicate entries.
        history = [
            old_row
            for old_row in history
            if int(old_row["epoch"]) != epoch
        ]
        history.append(row)

        history_df = (
            pd.DataFrame(history)
            .sort_values("epoch")
            .drop_duplicates(
                "epoch",
                keep="last",
            )
            .reset_index(drop=True)
        )

        improved = early.update(
            val_metrics["total"]
        )

        marker = "*" if improved else " "

        print(
            f"{marker} epoch={epoch:03d} "
            f"train={train_metrics['total']:.5f} "
            f"val={val_metrics['total']:.5f} "
            f"val_SI-SDR="
            f"{val_metrics['sisdr_db']:.3f} dB "
            f"time={elapsed:.1f}s"
        )

        # Save best model only when validation improves.
        if improved:
            best_state = {
                "epoch": epoch,
                "seed": seed,
                "variant": variant,
                "stage": stage_name,
                "model_state": model.state_dict(),
                "val_total": (
                    val_metrics["total"]
                ),
                "val_sisdr_db": (
                    val_metrics["sisdr_db"]
                ),
                "parameters": parameter_count(
                    model
                ),
                "config": asdict(CFG),
                "quantum_resources": (
                    quantum_resource_report()
                    if variant == "qnn"
                    else None
                ),
            }

            _atomic_torch_save(
                best_state,
                best_path,
            )

        # Save latest complete training state after every epoch.
        last_state = {
            "epoch": epoch,
            "seed": seed,
            "variant": variant,
            "stage": stage_name,

            "model_state": model.state_dict(),
            "optimizer_state": (
                optimizer.state_dict()
            ),
            "scheduler_state": (
                scheduler.state_dict()
            ),
            "scaler_state": (
                scaler.state_dict()
            ),

            "early_best": early.best,
            "early_bad_epochs": (
                early.bad_epochs
            ),

            "val_total": (
                val_metrics["total"]
            ),
            "val_sisdr_db": (
                val_metrics["sisdr_db"]
            ),

            "parameters": parameter_count(
                model
            ),
            "config": asdict(CFG),
            "history": (
                history_df.to_dict("records")
            ),

            "python_rng_state": (
                random.getstate()
            ),
            "numpy_rng_state": (
                np.random.get_state()
            ),
            "torch_rng_state": (
                torch.get_rng_state()
            ),
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available()
                else None
            ),
        }

        _atomic_torch_save(
            last_state,
            last_path,
        )

        _atomic_history_save(
            history_df,
            history_path,
        )

        if early.should_stop:
            print(
                "Early stopping triggered."
            )

            with open(
                completion_path,
                "w",
            ) as file:
                json.dump(
                    {
                        "seed": seed,
                        "stage": stage_name,
                        "variant": variant,
                        "last_history_epoch": epoch,
                        "reason": "early_stopping",
                    },
                    file,
                    indent=2,
                )

            break

    # Mark a max-epoch completion.
    final_history_df = _read_clean_history(
        history_path
    )

    if (
        not final_history_df.empty
        and int(
            final_history_df["epoch"].max()
        ) >= max_epochs
    ):
        with open(
            completion_path,
            "w",
        ) as file:
            json.dump(
                {
                    "seed": seed,
                    "stage": stage_name,
                    "variant": variant,
                    "last_history_epoch": int(
                        final_history_df[
                            "epoch"
                        ].max()
                    ),
                    "reason": "max_epochs",
                },
                file,
                indent=2,
            )

    # Always return the best validation model.
    best = _load_checkpoint_safely(
        best_path,
        DEVICE,
    )

    model.load_state_dict(
        best["model_state"]
    )

    return (
        model,
        final_history_df,
        best_path,
    )


print(
    "Resume-safe fit_model patch installed successfully."
)

## 10 — Run training

In [ ]:
CFG.shared_pretrain_epochs = 20
CFG.finetune_epochs = 35

CFG.early_stopping_patience = 6
CFG.min_delta = 1e-4

CFG.seeds = (17, 42, 123)
CFG.resume = True

# Must remain the same as the current experiment
CFG.experiment_name = "qmicrose_svqc_026m"

SPLIT_FRAMES = prepare_manifests()

TRAINING_SUMMARY = run_three_arm_protocol(
    SPLIT_FRAMES,
    seeds=CFG.seeds,
)

## 11 — Correct per-file metrics, noisy baselines, and a locked test set

In [ ]:
pip install pystoi

In [ ]:
EPS = 1e-8


def numpy_si_sdr(estimate: np.ndarray, reference: np.ndarray, eps: float = EPS) -> float:
    estimate = estimate - estimate.mean()
    reference = reference - reference.mean()
    scale = float(np.dot(estimate, reference)) / (float(np.dot(reference, reference)) + eps)
    target = scale * reference
    residual = estimate - target
    return float(10.0 * np.log10((np.sum(target ** 2) + eps) / (np.sum(residual ** 2) + eps)))


def reconstruction_snr(estimate: np.ndarray, reference: np.ndarray, eps: float = EPS) -> float:
    return float(10.0 * np.log10((np.mean(reference ** 2) + eps) / (np.mean((reference - estimate) ** 2) + eps)))


def compute_signal_metrics(reference: np.ndarray, estimate: np.ndarray, sample_rate: int = CFG.sample_rate):
    from pesq import pesq
    from pystoi import stoi

    length = min(len(reference), len(estimate))
    reference = np.asarray(reference[:length], dtype=np.float32)
    estimate = np.asarray(estimate[:length], dtype=np.float32)
    common_peak = max(float(np.max(np.abs(reference))), float(np.max(np.abs(estimate))), 1.0)
    reference = np.clip(reference / common_peak, -1.0, 1.0)
    estimate = np.clip(estimate / common_peak, -1.0, 1.0)

    result = {
        "si_sdr": numpy_si_sdr(estimate, reference),
        "reconstruction_snr": reconstruction_snr(estimate, reference),
    }
    try:
        result["pesq"] = float(pesq(sample_rate, reference, estimate, "wb"))
    except Exception:
        result["pesq"] = np.nan
    try:
        result["estoi"] = float(stoi(reference, estimate, sample_rate, extended=True))
    except Exception:
        result["estoi"] = np.nan
    return result


def _metric_worker(payload):
    reference, noisy, enhanced = payload
    noisy_metrics = compute_signal_metrics(reference, noisy)
    enhanced_metrics = compute_signal_metrics(reference, enhanced)
    row = {}
    for key, value in noisy_metrics.items():
        row[f"noisy_{key}"] = value
    for key, value in enhanced_metrics.items():
        row[f"enhanced_{key}"] = value
        row[f"delta_{key}"] = value - noisy_metrics[key]
    return row


def load_model_from_checkpoint(checkpoint_path: str | Path) -> QMicroSESVQC:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model = QMicroSESVQC(checkpoint["variant"]).to(DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    return model


@torch.no_grad()
def evaluate_checkpoint(
    checkpoint_path: str | Path,
    test_frame: pd.DataFrame,
    output_csv: str | Path,
    max_files_per_snr: int | None = CFG.max_test_files_per_snr,
    metric_workers: int = CFG.metric_workers,
):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    variant = checkpoint["variant"]
    seed = checkpoint["seed"]
    model = load_model_from_checkpoint(checkpoint_path)
    model.set_quantum_runtime(shots=None, readout_error=0.0, angle_noise_std=0.0)

    selected = test_frame[test_frame["snr_db"].isin(CFG.snr_values)].copy()
    selected = selected.sort_values(["snr_db", "sample_id"]).reset_index(drop=True)
    if max_files_per_snr is not None:
        selected = selected.groupby("snr_db", group_keys=False).head(max_files_per_snr).reset_index(drop=True)

    audio_payloads = []
    metadata_rows = []
    for _, row in tqdm(selected.iterrows(), total=len(selected), desc=f"Enhance {variant}/seed{seed}"):
        noisy = load_audio_mono(row["noisy_path"])
        clean = load_audio_mono(row["clean_path"])
        noisy_tensor = torch.from_numpy(noisy).unsqueeze(0).to(DEVICE)
        enhanced = model.enhance(noisy_tensor).squeeze(0).cpu().numpy().astype(np.float32)
        length = min(len(clean), len(noisy), len(enhanced))
        audio_payloads.append((clean[:length], noisy[:length], enhanced[:length]))
        metadata_rows.append({
            "sample_id": row["sample_id"],
            "snr_db": row["snr_db"],
            "variant": variant,
            "seed": seed,
            "checkpoint": str(checkpoint_path),
        })

    if metric_workers and metric_workers > 1:
        with ProcessPoolExecutor(max_workers=metric_workers) as executor:
            metric_rows = list(tqdm(
                executor.map(_metric_worker, audio_payloads),
                total=len(audio_payloads), desc="Metrics"
            ))
    else:
        metric_rows = [_metric_worker(payload) for payload in tqdm(audio_payloads, desc="Metrics")]

    result = pd.concat([pd.DataFrame(metadata_rows), pd.DataFrame(metric_rows)], axis=1)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_csv, index=False)
    print("Saved:", output_csv)
    return result


def evaluate_all_trained_models(training_summary: pd.DataFrame, test_frame: pd.DataFrame):
    all_results = []
    result_dir = Path("runs") / CFG.experiment_name / "test_results"
    for row in training_summary.itertuples(index=False):
        output_csv = result_dir / f"seed_{row.seed}_{row.variant}.csv"
        result = evaluate_checkpoint(row.checkpoint, test_frame, output_csv)
        all_results.append(result)
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(result_dir / "all_per_file_metrics.csv", index=False)
    return combined

TEST_RESULTS = evaluate_all_trained_models(TRAINING_SUMMARY, SPLIT_FRAMES["test"])

## 12 — Bootstrap confidence intervals and paired QNN-vs-MLP tests

In [ ]:
def bootstrap_mean_ci(values, repeats=CFG.bootstrap_repeats, seed=42, confidence=0.95):
    values = np.asarray(pd.Series(values).dropna(), dtype=np.float64)
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(repeats, len(values)))
    means = values[sample_indices].mean(axis=1)
    alpha = (1.0 - confidence) / 2.0
    return float(values.mean()), float(np.quantile(means, alpha)), float(np.quantile(means, 1.0 - alpha))


def summarise_test_results(results: pd.DataFrame):
    metrics = [
        "enhanced_pesq", "enhanced_estoi", "enhanced_si_sdr",
        "delta_pesq", "delta_estoi", "delta_si_sdr",
    ]
    rows = []
    for (variant, seed, snr), group in results.groupby(["variant", "seed", "snr_db"]):
        row = {"variant": variant, "seed": seed, "snr_db": snr, "n": len(group)}
        for metric in metrics:
            mean, low, high = bootstrap_mean_ci(group[metric], seed=int(seed) + 100)
            row[f"{metric}_mean"] = mean
            row[f"{metric}_ci_low"] = low
            row[f"{metric}_ci_high"] = high
        rows.append(row)
    summary = pd.DataFrame(rows)
    output = Path("runs") / CFG.experiment_name / "test_results" / "summary_with_ci.csv"
    summary.to_csv(output, index=False)
    return summary


def paired_qnn_vs_mlp(results: pd.DataFrame, metric: str = "delta_si_sdr"):
    from scipy.stats import wilcoxon

    paired = results[results["variant"].isin(["mlp", "qnn"])].pivot_table(
        index=["seed", "sample_id", "snr_db"], columns="variant", values=metric, aggfunc="first"
    ).dropna()
    differences = paired["qnn"] - paired["mlp"]
    mean_diff, ci_low, ci_high = bootstrap_mean_ci(differences, seed=2026)
    try:
        statistic, p_value = wilcoxon(differences, alternative="greater", zero_method="wilcox")
    except ValueError:
        statistic, p_value = np.nan, np.nan
    return pd.DataFrame([{
        "metric": metric,
        "paired_samples": len(differences),
        "qnn_minus_mlp_mean": mean_diff,
        "bootstrap_ci_low": ci_low,
        "bootstrap_ci_high": ci_high,
        "one_sided_wilcoxon_statistic": statistic,
        "one_sided_p_value": p_value,
    }])

TEST_SUMMARY = summarise_test_results(TEST_RESULTS)
display(TEST_SUMMARY)
for metric in ["delta_si_sdr", "delta_pesq", "delta_estoi"]:
    display(paired_qnn_vs_mlp(TEST_RESULTS, metric))

## 13 — Ideal, finite-shot, readout-noise, and control-noise audit

In [ ]:
@torch.no_grad()
def evaluate_quantum_noise_modes(
    checkpoint_path: str | Path,
    test_frame: pd.DataFrame,
    files_per_snr: int = 25,
    repeats: int = 3,
):
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    if checkpoint["variant"] != "qnn":
        raise ValueError("Noise audit requires a QNN checkpoint")
    model = load_model_from_checkpoint(checkpoint_path)

    selected = (
        test_frame[test_frame["snr_db"].isin(CFG.snr_values)]
        .sort_values(["snr_db", "sample_id"])
        .groupby("snr_db", group_keys=False)
        .head(files_per_snr)
        .reset_index(drop=True)
    )
    modes = [
        {"mode": "ideal", "shots": None, "readout_error": 0.0, "angle_noise_std": 0.0},
        {"mode": "shots_1024", "shots": 1024, "readout_error": 0.0, "angle_noise_std": 0.0},
        {"mode": "shots_256", "shots": 256, "readout_error": 0.0, "angle_noise_std": 0.0},
        {"mode": "shots_256_readout_1pct", "shots": 256, "readout_error": 0.01, "angle_noise_std": 0.0},
        {"mode": "shots_256_readout_3pct", "shots": 256, "readout_error": 0.03, "angle_noise_std": 0.0},
        {"mode": "shots_256_control_0p01rad", "shots": 256, "readout_error": 0.01, "angle_noise_std": 0.01},
    ]

    rows = []
    for mode in modes:
        mode_repeats = 1 if mode["mode"] == "ideal" else repeats
        for repeat in range(mode_repeats):
            seed_everything(10_000 + repeat)
            model.set_quantum_runtime(mode["shots"], mode["readout_error"], mode["angle_noise_std"])
            for _, sample in tqdm(selected.iterrows(), total=len(selected), desc=f"{mode['mode']} r{repeat}"):
                noisy = load_audio_mono(sample["noisy_path"])
                clean = load_audio_mono(sample["clean_path"])
                enhanced = model.enhance(torch.from_numpy(noisy).unsqueeze(0).to(DEVICE)).squeeze(0).cpu().numpy()
                length = min(len(clean), len(noisy), len(enhanced))
                noisy_metrics = compute_signal_metrics(clean[:length], noisy[:length])
                enhanced_metrics = compute_signal_metrics(clean[:length], enhanced[:length])
                rows.append({
                    "mode": mode["mode"],
                    "repeat": repeat,
                    "sample_id": sample["sample_id"],
                    "snr_db": sample["snr_db"],
                    "shots": mode["shots"],
                    "readout_error": mode["readout_error"],
                    "angle_noise_std": mode["angle_noise_std"],
                    **{f"enhanced_{key}": value for key, value in enhanced_metrics.items()},
                    **{f"delta_{key}": enhanced_metrics[key] - noisy_metrics[key] for key in enhanced_metrics},
                })

    model.set_quantum_runtime(None, 0.0, 0.0)
    result = pd.DataFrame(rows)
    output = Path("runs") / CFG.experiment_name / "quantum_noise_audit.csv"
    result.to_csv(output, index=False)
    print("Saved:", output)
    return result

# Example using the first trained QNN checkpoint:
QNN_CHECKPOINT = TRAINING_SUMMARY.query("variant == 'qnn'").iloc[0]["checkpoint"]
QUANTUM_NOISE_RESULTS = evaluate_quantum_noise_modes(QNN_CHECKPOINT, SPLIT_FRAMES["test"])

## 14 — Simulator latency and real-time-factor measurement

In [ ]:
@torch.no_grad()
def benchmark_inference(checkpoint_path: str | Path, seconds=(1, 4, 8), warmup=5, repeats=30):
    model = load_model_from_checkpoint(checkpoint_path)
    model.set_quantum_runtime(None, 0.0, 0.0)
    rows = []
    for duration in seconds:
        waveform = torch.randn(1, CFG.sample_rate * duration, device=DEVICE) * 0.02
        for _ in range(warmup):
            _ = model.enhance(waveform)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        timings = []
        for _ in range(repeats):
            start = time.perf_counter()
            _ = model.enhance(waveform)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            timings.append(time.perf_counter() - start)

        rows.append({
            "variant": model.variant,
            "device": str(DEVICE),
            "audio_seconds": duration,
            "mean_latency_ms": 1000 * np.mean(timings),
            "median_latency_ms": 1000 * np.median(timings),
            "p95_latency_ms": 1000 * np.quantile(timings, 0.95),
            "real_time_factor": np.mean(timings) / duration,
            "parameters": parameter_count(model),
            "simulator_only": True,
        })
    return pd.DataFrame(rows)

LATENCY_ROWS = []
for checkpoint in TRAINING_SUMMARY["checkpoint"]:
    LATENCY_ROWS.append(benchmark_inference(checkpoint))
LATENCY_RESULTS = pd.concat(LATENCY_ROWS, ignore_index=True)
display(LATENCY_RESULTS)
LATENCY_RESULTS.to_csv(Path("runs") / CFG.experiment_name / "latency.csv", index=False)

## 15 — Training and per-SNR result plots

In [ ]:
import matplotlib.pyplot as plt


def plot_training_history(history_csv: str | Path):
    history = pd.read_csv(history_csv)
    plt.figure(figsize=(8, 5))
    plt.plot(history["epoch"], history["train_total"], label="Train loss")
    plt.plot(history["epoch"], history["val_total"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Composite loss")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_metric_by_snr(results: pd.DataFrame, metric: str = "delta_si_sdr"):
    aggregated = results.groupby(["variant", "snr_db"])[metric].agg(["mean", "sem"]).reset_index()
    plt.figure(figsize=(8, 5))
    for variant, group in aggregated.groupby("variant"):
        plt.errorbar(group["snr_db"], group["mean"], yerr=1.96 * group["sem"], marker="o", label=variant)
    plt.xlabel("Input SNR [dB]")
    plt.ylabel(metric.replace("_", " ").upper())
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_metric_by_snr(TEST_RESULTS, "delta_si_sdr")
plot_metric_by_snr(TEST_RESULTS, "delta_pesq")
plot_metric_by_snr(TEST_RESULTS, "delta_estoi")

## 16 — Enhance and inspect a single recording

In [ ]:
@torch.no_grad()
def enhance_file(input_path: str, output_path: str, checkpoint_path: str | Path, return_aux: bool = False):
    model = load_model_from_checkpoint(checkpoint_path)
    waveform = load_audio_mono(input_path)
    tensor = torch.from_numpy(waveform).unsqueeze(0).to(DEVICE)
    if return_aux:
        enhanced, _, auxiliary = model(tensor, return_aux=True)
    else:
        enhanced = model.enhance(tensor)
        auxiliary = None
    enhanced = enhanced.squeeze(0).cpu().numpy()
    sf.write(output_path, enhanced, CFG.sample_rate)
    print("Saved:", output_path)
    return enhanced, auxiliary

enhanced, auxiliary = enhance_file(
    "noisy.wav",
    "enhanced_qmicrose_svqc.wav",
    TRAINING_SUMMARY.query("variant == 'qnn'").iloc[0]["checkpoint"],
    return_aux=True,
)

## 17 — Paper-quality decision rule

Do **not** select the final model by test-set performance. Choose hyperparameters using validation data only, freeze the experiment, and then evaluate the locked test manifest once.

A defensible quantum-benefit claim requires all of the following:

1. QNN and MLP have identical trainable parameter counts.
2. They use the same pretrained backbone, training batches, optimiser family, losses, and test files.
3. Results are averaged across at least three seeds.
4. QNN improves the primary metric, preferably SI-SDR improvement, with a paired bootstrap confidence interval above zero.
5. QNN does not materially degrade PESQ or ESTOI.
6. The advantage remains visible under finite-shot and modest readout/control noise.
7. Simulator latency is reported honestly as classical simulation latency, not physical quantum speed.

Suitable wording when the evidence supports it:

> A shallow six-qubit state-vector VQC provided a statistically significant improvement over an exactly parameter-matched classical MLP conditioner under a fixed sub-0.27M-parameter speech-enhancement backbone.

Wording to avoid:

- “quantum advantage” without physical-hardware evidence;
- “faster than classical computing” from simulator timing;
- “runs on a quantum computer”;
- selecting the best seed or best test condition after seeing test results.

## 18 — Recommended ablation matrix

Run these after the main three-arm experiment:

| Ablation | Values |
|---|---|
| Qubits | 4, 6, 8 |
| VQC layers | 2, 4 |
| Entanglement | none, linear, ring |
| Data re-uploading | off, on |
| Measurements | Z only, Z + neighbouring ZZ |
| Quantum location | after block 2, 3, or 5 |
| Shots | analytic, 1024, 256 |
| Readout error | 0%, 1%, 3% |
| Rotation-control noise | 0, 0.005, 0.01 radians |

For every architecture-changing ablation, recalculate the MLP control so its **core parameter budget matches the corresponding VQC**. Keep the locked manifests and report all attempted configurations, not only the winner.